# KNN model

In [55]:
import pandas as pd
import numpy as np

In [56]:
df = pd.read_csv("knn_customer_segments_revised.csv")

print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())

    age  annual_income_k  spending_score  visits_per_month  avg_purchase  \
0  58.0        16.016010       51.429550               8.0     22.876560   
1  25.0        13.697560       39.021830               9.0     50.571597   
2  47.0        17.556867        3.900580               8.0     37.093133   
3  52.0        37.522000       63.101190               5.0     83.490464   
4  60.0       133.644111       55.573214               4.0     19.495000   

   online_orders_pct  loyalty_years  discount_usage_pct customer_segment  
0          40.987764       1.192108           72.196300          premium  
1          43.748737       4.817224           41.984260          premium  
2          54.817156       9.851475           92.406077          regular  
3          11.312465       0.928982           67.661425           budget  
4          47.628909       5.195361           69.078600          regular  
<class 'pandas.DataFrame'>
RangeIndex: 910 entries, 0 to 909
Data columns (total 9 columns):


## little data cleaning

In [57]:
# unreal values replace with nan
df.loc[~df["age"].between(18, 100), "age"] = np.nan
df.loc[df["annual_income_k"] <= 0, "annual_income_k"] = np.nan
df.loc[~df["spending_score"].between(0, 100), "spending_score"] = np.nan
df.loc[df["visits_per_month"] <= 0, "visits_per_month"] = np.nan
df.loc[df["avg_purchase"] <= 0, "avg_purchase"] = np.nan
df.loc[~df["online_orders_pct"].between(0, 100), "online_orders_pct"] = np.nan
df.loc[df["loyalty_years"] < 0, "loyalty_years"] = np.nan
df.loc[~df["discount_usage_pct"].between(0, 100), "discount_usage_pct"] = np.nan

In [58]:
# fill missing
num_cols = df.select_dtypes(include="number").columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# del duplicates
df = df.drop_duplicates().copy()

df = df.copy()

# print(df.isnull().sum()) # o ta


In [59]:
# check dataset after datacleaning
print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())

    age  annual_income_k  spending_score  visits_per_month  avg_purchase  \
0  58.0        16.016010       51.429550               8.0     22.876560   
1  25.0        13.697560       39.021830               9.0     50.571597   
2  47.0        17.556867        3.900580               8.0     37.093133   
3  52.0        37.522000       63.101190               5.0     83.490464   
4  60.0       133.644111       55.573214               4.0     19.495000   

   online_orders_pct  loyalty_years  discount_usage_pct customer_segment  
0          40.987764       1.192108           72.196300          premium  
1          43.748737       4.817224           41.984260          premium  
2          54.817156       9.851475           92.406077          regular  
3          11.312465       0.928982           67.661425           budget  
4          47.628909       5.195361           69.078600          regular  
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 9 columns):


# Baseline KNN model

In [83]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

X = df[['annual_income_k', "spending_score", 'visits_per_month', "avg_purchase", 'online_orders_pct', "loyalty_years", "discount_usage_pct"]]
y = df['customer_segment']

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
knn = KNeighborsClassifier(n_neighbors=5)

#train and predict
knn.fit(X_train, y_train)
print(knn.score(X_test, y_test)) # 0.38333333333333336

# classifcation report
print(classification_report(y_test, knn.predict(X_test)))

0.38333333333333336
              precision    recall  f1-score   support

      budget       0.32      0.40      0.35        43
     premium       0.37      0.39      0.38        56
     regular       0.44      0.37      0.40        81

    accuracy                           0.38       180
   macro avg       0.38      0.39      0.38       180
weighted avg       0.39      0.38      0.38       180



## pipeline

In [84]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

pipe = Pipeline([
    # ('scaler', StandardScaler()),
    # ("minmax_scaler", MinMaxScaler()),
    ("robust_scaler", RobustScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
pipe.fit(X_train, y_train)
print(pipe.score(X_test, y_test)) # 0.49444444444444446 robust scalerda oshdi

0.49444444444444446


In [85]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for k in range(1, 35, 2):
    p = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    ball = cross_val_score(p, X_train, y_train, cv=cv, n_jobs=-1)
    scores.append((k, ball))

print(max(scores, key=lambda t: t[1].mean()))
# (19, array([0.40972222, 0.54861111, 0.48611111, 0.5       , 0.49305556])) k = 19 eng zori chiqibti

(19, array([0.40972222, 0.54861111, 0.48611111, 0.5       , 0.49305556]))


# Grid search

In [87]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# pipeline
pipe_grid = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

# define hyperparameters
param_grid = {
    'scaler': [StandardScaler(), RobustScaler(), MinMaxScaler()],
    'knn__n_neighbors' : list(range(3, 40, 2)),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

# Search for the combination with the best average macro F1
grid = GridSearchCV(
    pipe_grid,
    param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)

print('best params - ', grid.best_params_)

print('best score - ', grid.best_score_)

best_model = grid.best_estimator_
pred = best_model.predict(X_test)

print("test accuracy:", accuracy_score(y_test, pred))
print("classification report: ", classification_report(y_test, pred))
print("confusion matrix: ", confusion_matrix(y_test, pred))

# best score -  0.4635333863681258
# test accuracy: 0.5166666666666667

Fitting 5 folds for each of 228 candidates, totalling 1140 fits
best params -  {'knn__metric': 'euclidean', 'knn__n_neighbors': 15, 'knn__weights': 'uniform', 'scaler': RobustScaler()}
best score -  0.4635333863681258
test accuracy: 0.5166666666666667
classification report:                precision    recall  f1-score   support

      budget       0.47      0.37      0.42        43
     premium       0.59      0.43      0.49        56
     regular       0.50      0.65      0.57        81

    accuracy                           0.52       180
   macro avg       0.52      0.48      0.49       180
weighted avg       0.52      0.52      0.51       180

confusion matrix:  [[16  5 22]
 [ 2 24 30]
 [16 12 53]]


# checking , optimization of accuracy

In [88]:
print(df.columns.tolist())
print(df["customer_segment"].value_counts())

['age', 'annual_income_k', 'spending_score', 'visits_per_month', 'avg_purchase', 'online_orders_pct', 'loyalty_years', 'discount_usage_pct', 'customer_segment', 'monthly_spend', 'visits_x_loyalty']
customer_segment
regular    405
premium    279
budget     216
Name: count, dtype: int64


In [89]:
# feature engineering
df["monthly_spend_est"] = (df["visits_per_month"] * df["avg_purchase"])
df["online_spend_est"] = (df["monthly_spend_est"] * df["online_orders_pct"] / 100)
df["discount_spend_est"] = (df["monthly_spend_est"] * df["discount_usage_pct"] / 100)
df["spend_to_income_ratio"] = (df["avg_purchase"] / (df["annual_income_k"] * 1000 / 12))

# original features + age + engineered features
features = [
    "age",
    "annual_income_k",
    "spending_score",
    "visits_per_month",
    "avg_purchase",
    "online_orders_pct",
    "loyalty_years",
    "discount_usage_pct",
    "monthly_spend_est",
    "online_spend_est",
    "discount_spend_est",
    "spend_to_income_ratio"
]

X_exp = df[features]
y_exp = df["customer_segment"]

# keep the same train/test rows as your previous experiment
X_train_exp = X_exp.loc[X_train.index]
X_test_exp = X_exp.loc[X_test.index]

print(X_train_exp.shape, X_test_exp.shape)


(720, 12) (180, 12)


In [90]:

from sklearn.model_selection import cross_val_score
from sklearn.base import clone
import pandas as pd

original = [
    "annual_income_k",
    "spending_score",
    "visits_per_month",
    "avg_purchase",
    "online_orders_pct",
    "loyalty_years",
    "discount_usage_pct"
]

experiments = {
    "original": original,
    "original + age": original + ["age"],
    "all engineered": features,
    "without income": [f for f in features if f != "annual_income_k"],
    "without spending_score": [f for f in features if f != "spending_score"],
    "without visits": [f for f in features if f != "visits_per_month"],
    "without avg_purchase": [f for f in features if f != "avg_purchase"],
    "without online_orders": [f for f in features if f != "online_orders_pct"],
    "without loyalty": [f for f in features if f != "loyalty_years"],
    "without discount_usage": [f for f in features if f != "discount_usage_pct"],
    "without age": [f for f in features if f != "age"]
}

results = []

for name, cols in experiments.items():
    model = clone(best_model)

    scores = cross_val_score(
        model,
        X_train_exp[cols],
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    results.append({
        "experiment": name,
        "features": len(cols),
        "mean_f1": scores.mean(),
        "std_f1": scores.std()
    })

results_df = pd.DataFrame(results).sort_values(
    "mean_f1", ascending=False
)

print(results_df.to_string(index=False))


            experiment  features  mean_f1   std_f1
 without online_orders        11 0.477951 0.037400
        all engineered        12 0.466813 0.024075
           without age        11 0.466632 0.040710
  without avg_purchase        11 0.464501 0.037174
              original         7 0.463533 0.038961
        without income        11 0.463527 0.025451
        original + age         8 0.456637 0.046209
without spending_score        11 0.454798 0.041332
        without visits        11 0.441016 0.010238
without discount_usage        11 0.420274 0.016387
       without loyalty        11 0.399161 0.041949


In [91]:
best_features = experiments[
    results_df.iloc[0]["experiment"]
]

final_model = clone(best_model)

final_model.fit(X_train_exp[best_features], y_train)

pred = final_model.predict(X_test_exp[best_features])

print("Features:", best_features)
print("Test accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

Features: ['age', 'annual_income_k', 'spending_score', 'visits_per_month', 'avg_purchase', 'loyalty_years', 'discount_usage_pct', 'monthly_spend_est', 'online_spend_est', 'discount_spend_est', 'spend_to_income_ratio']
Test accuracy: 0.5055555555555555
              precision    recall  f1-score   support

      budget       0.41      0.37      0.39        43
     premium       0.60      0.46      0.53        56
     regular       0.50      0.60      0.55        81

    accuracy                           0.51       180
   macro avg       0.50      0.48      0.49       180
weighted avg       0.51      0.51      0.50       180

[[16  7 20]
 [ 1 26 29]
 [22 10 49]]


## New features kop yordam bermadi!!! / endi overlapping solve

In [93]:
# Rebuild X using all available features
all_features = [
    "age",
    "annual_income_k",
    "spending_score",
    "visits_per_month",
    "avg_purchase",
    "online_orders_pct",
    "loyalty_years",
    "discount_usage_pct",
    "monthly_spend",
    "visits_x_loyalty"
]

X_all = df[all_features]

X_train = X_all.loc[X_train.index]
X_test = X_all.loc[X_test.index]

print("Training features:", X_train.columns.tolist())
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)


Training features: ['age', 'annual_income_k', 'spending_score', 'visits_per_month', 'avg_purchase', 'online_orders_pct', 'loyalty_years', 'discount_usage_pct', 'monthly_spend', 'visits_x_loyalty']
Training shape: (720, 10)
Test shape: (180, 10)


In [94]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

# test different sets of features
feature_sets = {
    "original": [
        "annual_income_k", "spending_score", "visits_per_month",
        "avg_purchase", "online_orders_pct", "loyalty_years",
        "discount_usage_pct"
    ],
    "remove_online": [
        "annual_income_k", "spending_score", "visits_per_month",
        "avg_purchase", "loyalty_years", "discount_usage_pct"
    ],
    "existing_engineered": [
        "age", "annual_income_k", "spending_score",
        "visits_per_month", "avg_purchase", "loyalty_years",
        "discount_usage_pct", "monthly_spend", "visits_x_loyalty"
    ],
    "spending_features": [
        "annual_income_k", "spending_score", "avg_purchase",
        "monthly_spend", "discount_usage_pct"
    ],
    "behavior_features": [
        "spending_score", "visits_per_month", "avg_purchase",
        "loyalty_years", "discount_usage_pct", "monthly_spend",
        "visits_x_loyalty"
    ]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "scaler": [StandardScaler(), RobustScaler(), MinMaxScaler()],
    "knn__n_neighbors": list(range(3, 40, 2)),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}

results = []

for name, cols in feature_sets.items():
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier())
    ])

    search = GridSearchCV(
        model,
        param_grid,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    search.fit(X_train[cols], y_train)

    results.append({
        "features": name,
        "cv_macro_f1": search.best_score_,
        "best_params": search.best_params_,
        "model": search.best_estimator_,
        "columns": cols
    })

results.sort(key=lambda x: x["cv_macro_f1"], reverse=True)

for r in results:
    print(r["features"], round(r["cv_macro_f1"], 4))
    print(r["best_params"])
    print()


behavior_features 0.4791
{'knn__metric': 'manhattan', 'knn__n_neighbors': 27, 'knn__weights': 'uniform', 'scaler': RobustScaler()}

existing_engineered 0.4762
{'knn__metric': 'euclidean', 'knn__n_neighbors': 25, 'knn__weights': 'uniform', 'scaler': RobustScaler()}

remove_online 0.4758
{'knn__metric': 'euclidean', 'knn__n_neighbors': 17, 'knn__weights': 'uniform', 'scaler': RobustScaler()}

original 0.4635
{'knn__metric': 'euclidean', 'knn__n_neighbors': 15, 'knn__weights': 'uniform', 'scaler': RobustScaler()}

spending_features 0.4048
{'knn__metric': 'euclidean', 'knn__n_neighbors': 13, 'knn__weights': 'uniform', 'scaler': RobustScaler()}



In [95]:

winner = results[0]

best_features = winner["columns"]
best_model = winner["model"]

pred = best_model.predict(X_test[best_features])

print("Winning feature set:", winner["features"])
print("CV macro F1:", winner["cv_macro_f1"])
print("Test accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


Winning feature set: behavior_features
CV macro F1: 0.4791377157866476
Test accuracy: 0.5444444444444444
              precision    recall  f1-score   support

      budget       0.45      0.42      0.43        43
     premium       0.68      0.48      0.56        56
     regular       0.53      0.65      0.59        81

    accuracy                           0.54       180
   macro avg       0.55      0.52      0.53       180
weighted avg       0.56      0.54      0.54       180

